# ECHOSCOPE AI — Beyond the Surface of Data
### A reproducible research notebook on hidden statistical structures and machine-learning reliability

**Research question:** Can statistical analysis and machine learning uncover hidden structures, anomalies, and misleading relationships in datasets that appear statistically similar?

**Experiments**
1. Statistical illusion: Anscombe's quartet
2. Invisible relationships: linear, monotonic, and nonlinear association
3. Hidden dimensions: PCA and explained variance
4. The outlier effect: IQR, Z-score, and Isolation Forest
5. The generalization trap: KNN, Linear Regression, and Ridge Regression
6. Correlation is not causation: a confounding-variable simulation
7. Stability under noise and sample-size changes

**How to use:** Run cells from top to bottom. The notebook creates all data synthetically with fixed random seeds, so results are reproducible and no external downloads are required.

> Synthetic experiments demonstrate statistical principles; they are not evidence about a real-world population.

## 0. Environment and reproducibility
Install missing packages if needed: `pip install numpy pandas matplotlib seaborn scipy scikit-learn`.

In [ ]:
import sys, warnings, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor, KNeighborsRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

warnings.filterwarnings("ignore")
SEED = 42
rng = np.random.default_rng(SEED)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.titleweight"] = "bold"

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

## 1. Statistical illusion — Anscombe's Quartet

Anscombe's quartet contains four datasets with nearly identical descriptive statistics and linear-regression summaries, yet very different shapes. It demonstrates why numerical summaries must be paired with visualization.

**Questions**
- Do the four datasets have similar means, variances, correlations, and regression lines?
- Do their scatter plots look alike?
- What information is lost when we report only summary statistics?

In [ ]:
# Anscombe's quartet is bundled with seaborn; no network access is needed.
anscombe = sns.load_dataset("anscombe")
summary = (anscombe.groupby("dataset")
           .apply(lambda g: pd.Series({
               "x_mean": g.x.mean(), "y_mean": g.y.mean(),
               "x_variance": g.x.var(), "y_variance": g.y.var(),
               "correlation": g.x.corr(g.y),
               "slope": np.polyfit(g.x, g.y, 1)[0],
               "intercept": np.polyfit(g.x, g.y, 1)[1]
           }), include_groups=False))
display(summary.round(3))

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for ax, (name, group) in zip(axes.flat, anscombe.groupby("dataset")):
    sns.regplot(data=group, x="x", y="y", ax=ax,
                ci=None, scatter_kws={"s": 55}, line_kws={"color": "crimson"})
    ax.set_title(f"Dataset {name}")
plt.suptitle("Anscombe's Quartet: similar summaries, different structures", y=1.02, fontsize=15)
plt.tight_layout()
plt.show()

### Interpretation
Dataset I is approximately linear. Dataset II contains a curved relationship. Dataset III is strongly influenced by an outlier. Dataset IV has most x-values concentrated at one location, with one high-leverage point. A single correlation or regression line cannot describe these structures adequately.

## 2. Invisible relationships — correlation is not the whole story

Pearson correlation measures **linear association**. Spearman correlation measures **monotonic association using ranks**. Neither is a universal detector of every possible relationship.

We compare:
- Linear positive association
- Monotonic but nonlinear association
- U-shaped association
- Circular structure

In [ ]:
n = 500
x = np.linspace(-3, 3, n)
linear_y = 2*x + rng.normal(0, 1.2, n)
monotonic_y = np.exp(x/1.5) + rng.normal(0, 0.35, n)
u_shape_y = x**2 + rng.normal(0, 0.6, n)
theta = rng.uniform(0, 2*np.pi, n)
circle_x = np.cos(theta) + rng.normal(0, 0.06, n)
circle_y = np.sin(theta) + rng.normal(0, 0.06, n)

relationships = {
    "Linear": (x, linear_y),
    "Monotonic nonlinear": (x, monotonic_y),
    "U-shaped": (x, u_shape_y),
    "Circular": (circle_x, circle_y)
}
rows = []
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for ax, (label, (xx, yy)) in zip(axes.flat, relationships.items()):
    ax.scatter(xx, yy, alpha=.55, s=16)
    ax.set_title(label)
    rows.append({
        "relationship": label,
        "pearson_r": stats.pearsonr(xx, yy).statistic,
        "spearman_rho": stats.spearmanr(xx, yy).statistic
    })
plt.tight_layout()
plt.show()
display(pd.DataFrame(rows).set_index("relationship").round(3))

### Interpretation guide
- Pearson near zero does **not** prove independence. The U-shaped and circular examples can have weak linear correlation despite clear structure.
- Spearman can capture monotonic nonlinear relationships, but it can also be near zero for non-monotonic patterns.
- Always plot the data and use domain knowledge. Correlation describes association, not causation.

## 3. Hidden dimensions — PCA and dimensionality reduction

Principal Component Analysis (PCA) creates new orthogonal variables (principal components) that capture decreasing amounts of variance.

**Important steps**
1. Standardize features when they use different units or scales.
2. Fit PCA on training data only in predictive workflows.
3. Inspect explained variance.
4. Choose the number of components based on a justified trade-off, not an automatic rule.

Here we create correlated high-dimensional features from a smaller number of latent signals.

In [ ]:
n_samples, n_features, n_latent = 600, 12, 3
latent = rng.normal(size=(n_samples, n_latent))
loadings = rng.normal(size=(n_latent, n_features))
X_high = latent @ loadings + rng.normal(scale=0.35, size=(n_samples, n_features))
feature_names = [f"feature_{i+1}" for i in range(n_features)]
X_df = pd.DataFrame(X_high, columns=feature_names)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_df)
pca = PCA().fit(X_scaled)
cum_var = np.cumsum(pca.explained_variance_ratio_)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].bar(np.arange(1, n_features+1), pca.explained_variance_ratio_)
axes[0].set(title="Variance explained by each component", xlabel="Principal component", ylabel="Explained variance ratio")
axes[1].plot(np.arange(1, n_features+1), cum_var, marker="o")
axes[1].axhline(.90, color="crimson", linestyle="--", label="90% reference")
axes[1].set(title="Cumulative explained variance", xlabel="Number of components", ylabel="Cumulative variance")
axes[1].set_ylim(0, 1.03); axes[1].legend()
plt.tight_layout(); plt.show()

n90 = int(np.argmax(cum_var >= .90) + 1)
print(f"Components needed to retain at least 90% variance: {n90} of {n_features}")
X_pca_2d = PCA(n_components=2).fit_transform(X_scaled)
plt.figure(figsize=(7, 5))
plt.scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], alpha=.55, s=18)
plt.title("High-dimensional observations projected onto first two PCs")
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.show()

### PCA concepts to remember
- **Eigenvector:** direction of a principal component.
- **Eigenvalue:** variance captured along that direction.
- **Explained variance ratio:** fraction of total variance represented by a component.
- PCA is unsupervised: it does not use a target label.
- PCA preserves directions of high variance, not necessarily the directions most useful for predicting a target.

## 4. The outlier effect — IQR, Z-score, Isolation Forest

An outlier is an observation that differs markedly from the majority. An unusual value is not automatically an error: it may be a valid rare event.

We compare three approaches:
- IQR rule: univariate, based on quartiles.
- Z-score: univariate, based on distance from mean in standard deviations.
- Isolation Forest: multivariate anomaly-detection algorithm.

In [ ]:
# Create a two-dimensional dataset with injected anomalies.
X_normal = rng.normal(loc=0, scale=1, size=(500, 2))
X_anomaly = rng.uniform(low=-5, high=5, size=(25, 2))
X_out = np.vstack([X_normal, X_anomaly])
truth = np.array([0]*len(X_normal) + [1]*len(X_anomaly))
out_df = pd.DataFrame(X_out, columns=["x1", "x2"])

# IQR flags on each feature; a point is flagged if either feature is outside bounds.
iqr_flags = np.zeros(len(out_df), dtype=bool)
for col in out_df:
    q1, q3 = out_df[col].quantile([.25, .75])
    iqr = q3-q1
    lower, upper = q1-1.5*iqr, q3+1.5*iqr
    iqr_flags |= ~out_df[col].between(lower, upper)

# Z-score: flag if either absolute z-score exceeds 3.
z_scores = np.abs(stats.zscore(out_df, axis=0))
z_flags = (z_scores > 3).any(axis=1)

iso = IsolationForest(contamination=len(X_anomaly)/len(X_out), random_state=SEED)
iso_flags = iso.fit_predict(out_df) == -1

def detection_report(name, flags):
    tp = np.sum(flags & (truth == 1))
    fp = np.sum(flags & (truth == 0))
    fn = np.sum(~flags & (truth == 1))
    precision = tp/(tp+fp) if tp+fp else 0
    recall = tp/(tp+fn) if tp+fn else 0
    return {"method": name, "flagged": int(flags.sum()), "precision": precision, "recall": recall}

display(pd.DataFrame([
    detection_report("IQR (either feature)", iqr_flags),
    detection_report("Z-score (>|3|)", z_flags),
    detection_report("Isolation Forest", iso_flags)
]).set_index("method").round(3))

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, flags, title in zip(axes, [iqr_flags, z_flags, iso_flags],
                            ["IQR rule", "Z-score rule", "Isolation Forest"]):
    ax.scatter(X_out[~flags, 0], X_out[~flags, 1], s=18, alpha=.5, label="Not flagged")
    ax.scatter(X_out[flags, 0], X_out[flags, 1], s=28, alpha=.8, label="Flagged")
    ax.set_title(title); ax.set_xlabel("x1"); ax.set_ylabel("x2")
    ax.legend()
plt.tight_layout(); plt.show()

### Important caution
The synthetic anomalies were deliberately generated from a broad uniform distribution, so they are easier to detect than subtle real anomalies. Thresholds and contamination assumptions should be justified for real applications. Never delete outliers automatically without checking whether they are data errors or meaningful rare events.

## 5. The generalization trap — KNN, Linear Regression, and Ridge Regression

A model should perform well on **unseen observations**, not merely memorize its training data. We create a noisy nonlinear signal and compare:
- KNN regression with different neighborhood sizes
- Linear regression
- Ridge regression with different regularization strengths

A very small K in KNN can fit noise; a very large K can oversmooth the relationship.

In [ ]:
X = rng.uniform(-3, 3, size=(700, 1))
y = np.sin(1.5*X[:, 0]) + 0.25*X[:, 0] + rng.normal(0, .35, 700)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=.25, random_state=SEED
)

models = {
    "KNN (k=1)": KNeighborsRegressor(n_neighbors=1),
    "KNN (k=15)": KNeighborsRegressor(n_neighbors=15),
    "KNN (k=60)": KNeighborsRegressor(n_neighbors=60),
    "Linear Regression": LinearRegression(),
    "Ridge (alpha=10)": Ridge(alpha=10)
}
results = []
x_grid = np.linspace(X.min(), X.max(), 400).reshape(-1, 1)
plt.figure(figsize=(11, 6))
plt.scatter(X_train[:, 0], y_train, s=13, alpha=.25, label="Training observations")
plt.scatter(X_test[:, 0], y_test, s=18, alpha=.45, label="Test observations")
for name, model in models.items():
    model.fit(X_train, y_train)
    pred_train, pred_test = model.predict(X_train), model.predict(X_test)
    results.append({
        "model": name,
        "train_RMSE": mean_squared_error(y_train, pred_train)**.5,
        "test_RMSE": mean_squared_error(y_test, pred_test)**.5,
        "test_MAE": mean_absolute_error(y_test, pred_test),
        "test_R2": r2_score(y_test, pred_test)
    })
    plt.plot(x_grid[:, 0], model.predict(x_grid), label=name, linewidth=2)
plt.title("Comparing model flexibility on a nonlinear signal")
plt.xlabel("x"); plt.ylabel("y"); plt.legend(ncol=2); plt.show()
display(pd.DataFrame(results).set_index("model").sort_values("test_RMSE").round(3))

### How to interpret the comparison
- Compare training and test error together.
- A very low training error with substantially worse test error suggests overfitting.
- High errors on both training and test sets can indicate underfitting.
- Model choice should be based on validation or cross-validation, then assessed once on a held-out test set.

## 6. Correlation is not causation — a confounding-variable simulation

A confounder influences both a predictor and an outcome. This can create or distort an observed association.

We simulate:
- `study_time` and `exam_score`
- `prior_preparation` influences both
- A naive association may partly reflect prior preparation

This is a teaching simulation, not a causal estimate from real people.

In [ ]:
n = 1200
prior_preparation = rng.normal(size=n)
study_time = 2.0*prior_preparation + rng.normal(scale=1.0, size=n)
exam_score = 3.0*prior_preparation + 0.5*study_time + rng.normal(scale=1.2, size=n)
conf = pd.DataFrame({
    "prior_preparation": prior_preparation,
    "study_time": study_time,
    "exam_score": exam_score
})

naive_r = conf["study_time"].corr(conf["exam_score"])
partial_model = LinearRegression().fit(
    conf[["study_time", "prior_preparation"]], conf["exam_score"]
)
print(f"Naive Pearson correlation (study time vs exam score): {naive_r:.3f}")
print("Multiple regression coefficients:")
display(pd.Series(partial_model.coef_, index=["study_time", "prior_preparation"]).to_frame("coefficient"))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.regplot(data=conf, x="study_time", y="exam_score", ax=axes[0],
            scatter_kws={"alpha": .25, "s": 15})
axes[0].set_title("Naive bivariate relationship")
sc = axes[1].scatter(conf.study_time, conf.exam_score, c=conf.prior_preparation,
                     cmap="viridis", alpha=.6, s=18)
axes[1].set_title("Same relationship, colored by prior preparation")
axes[1].set_xlabel("Study time"); axes[1].set_ylabel("Exam score")
plt.colorbar(sc, ax=axes[1], label="Prior preparation")
plt.tight_layout(); plt.show()

### Interpretation
The bivariate correlation mixes the association of study time with the influence of prior preparation. Adding a measured confounder to a regression model can help separate conditional associations, but observational adjustment alone does not prove causality. Causal claims require assumptions and an appropriate study design.

## 7. Stability — what changes when noise and sample size change?

A finding from a small sample may be unstable. More observations can reduce sampling variability, while stronger noise can make patterns harder to detect.

We estimate the correlation repeatedly for different sample sizes and noise levels.

In [ ]:
sample_sizes = [30, 100, 500]
noise_levels = [0.3, 1.0, 2.0]
records = []
for n in sample_sizes:
    for noise in noise_levels:
        for rep in range(150):
            xx = rng.normal(size=n)
            yy = 0.8*xx + rng.normal(scale=noise, size=n)
            records.append({"sample_size": n, "noise_sd": noise,
                            "estimated_r": np.corrcoef(xx, yy)[0, 1]})
stability = pd.DataFrame(records)

g = sns.FacetGrid(stability, row="noise_sd", col="sample_size",
                  margin_titles=True, height=2.5, aspect=1.15)
g.map_dataframe(sns.histplot, x="estimated_r", bins=18, stat="density")
g.set_axis_labels("Estimated Pearson correlation", "Density")
g.fig.suptitle("Sampling stability across sample sizes and noise levels", y=1.03)
plt.show()

display(stability.groupby(["sample_size", "noise_sd"])["estimated_r"]
        .agg(["mean", "std"]).round(3))

## 8. Research-style findings template

Use this table after running the notebook. Replace the prompts with your actual observations; do not invent results.

| Experiment | What to report |
|---|---|
| Statistical illusion | Which datasets had similar summaries but different visual structures? |
| Correlation | Which patterns were poorly represented by Pearson correlation? |
| PCA | How many components reached your chosen explained-variance threshold? |
| Anomaly detection | Which method had the best precision/recall in this simulation? |
| Generalization | Which model had the lowest test RMSE? Was there a train-test gap? |
| Confounding | How did the bivariate association compare with the adjusted model? |
| Stability | How did sample size and noise affect the spread of estimated correlations? |

### Suggested conclusion structure
1. State the central research question.
2. Summarize the strongest result from each experiment.
3. Explain why visualization and validation matter.
4. Identify limitations of synthetic data.
5. Propose a real dataset and follow-up experiment.

## 9. Further challenges

1. Change the random seed and assess whether the main conclusions remain similar.
2. Add a heavy-tailed distribution and compare mean, median, standard deviation, and IQR.
3. Compare Pearson, Spearman, and mutual information for nonlinear relationships.
4. Fit PCA only on training data using a `Pipeline`, then compare downstream predictive performance.
5. Tune KNN's `n_neighbors` and Ridge's `alpha` with `GridSearchCV` and 5-fold cross-validation.
6. Add a second confounder and explain how the interpretation changes.

### Example: Grid search with a leakage-safe pipeline

In [ ]:
# Demonstration: scaling is fitted separately inside each training fold.
X_reg = rng.normal(size=(500, 8))
true_w = np.array([2.0, -1.5, 0.0, 0.0, 0.8, 0.0, 0.0, 0.0])
y_reg = X_reg @ true_w + rng.normal(scale=1.5, size=500)

Xtr, Xte, ytr, yte = train_test_split(X_reg, y_reg, test_size=.2, random_state=SEED)

pipe = Pipeline([
    ("scale", StandardScaler()),
    ("ridge", Ridge())
])
param_grid = {"ridge__alpha": [0.01, 0.1, 1, 10, 100]}
cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
search = GridSearchCV(pipe, param_grid, scoring="neg_root_mean_squared_error",
                      cv=cv, n_jobs=-1)
search.fit(Xtr, ytr)

best_model = search.best_estimator_
test_pred = best_model.predict(Xte)
print("Best parameters:", search.best_params_)
print("Best CV RMSE:", -search.best_score_)
print("Held-out test RMSE:", mean_squared_error(yte, test_pred)**.5)
print("Held-out test R2:", r2_score(yte, test_pred))

## 10. Key references and further reading

- Anscombe, F. J. (1973). Graphs in statistical analysis. *The American Statistician*, 27(1), 17–21.
- Jolliffe, I. T., & Cadima, J. (2016). Principal component analysis: A review and recent developments. *Philosophical Transactions of the Royal Society A*, 374.
- Breiman, L. (2001). Random forests. *Machine Learning*, 45, 5–32.
- scikit-learn documentation: https://scikit-learn.org/stable/
- SciPy statistics documentation: https://docs.scipy.org/doc/scipy/reference/stats.html

**Reproducibility note:** The synthetic data are generated within the notebook. Library versions and random seeds are printed or fixed where applicable.